# 弗淼 AIGC · Colab 接入

先连接 GPU 运行时，再依次运行以下单元。模型选择沿用网站生成的接入命令。权重保存到 Drive 的 `FumiaoAIGC`，运行环境、接入凭据和任务文件留在实例本地。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# @title 安装并接入
Drive缓存上限GB = 160 # @param {type:"number"}
# 请设置为不超过 Drive 实际剩余空间的值；此处不是购买存储。
import os, getpass, subprocess, shlex, re
from pathlib import Path
assert Path("/content/drive").is_mount(), "请先完成 Drive 挂载"
assert 1 <= Drive缓存上限GB <= 30000
connect_command = getpass.getpass("粘贴平台生成的完整接入命令：")
parts = []
code = ""
try:
    parts = shlex.split(connect_command)
    if len(parts) != 11 or parts[:2] != ["curl", "-fsSL"] or parts[3:6] != ["-o", "/tmp/fumiao-install.sh", "&&"] or parts[7:9] != ["bash", "/tmp/fumiao-install.sh"]:
        raise ValueError("请粘贴平台生成的完整接入命令")
    site = parts[6].removeprefix("FUMIAO_SITE=")
    if not re.fullmatch(r"https://nvmes\.com(?:/__gpu_check_[a-f0-9]+)?", site) or parts[2] != site + "/connect/install.sh":
        raise ValueError("接入地址不匹配，请重新复制平台命令")
    models = parts[9].removeprefix("--models=")
    code = parts[10]
    if not parts[9].startswith("--models=") or not set(models.split(",")) <= {"minimax-h3", "breeze-tts-2"} or not re.fullmatch(r"[A-Za-z0-9_-]{30,128}", code):
        raise ValueError("接入参数无效，请重新复制平台命令")
    env = dict(os.environ, FUMIAO_SITE=site, FUMIAO_HOME="/content/fumiao-worker", FUMIAO_CACHE_ROOT="/opt",
               FUMIAO_PERSIST_ROOT="/content/drive/MyDrive/FumiaoAIGC",
               FUMIAO_PERSIST_QUOTA_BYTES=str(int(Drive缓存上限GB * 1_000_000_000)))
    env.pop("FUMIAO_NO_FOLLOW", None)
    if subprocess.run(["curl", "-fsSL", parts[2], "-o", "/tmp/fumiao-install.sh"]).returncode:
        raise RuntimeError("下载接入脚本失败，请稍后重试")
    # 用参数数组执行；接入码不会写入笔记本正文或错误信息。
    result = subprocess.run(["bash", "/tmp/fumiao-install.sh", "--models=" + models, code], env=env)
    if result.returncode:
        raise RuntimeError(f"接入脚本退出（{result.returncode}），请检查上方安装日志")
finally:
    connect_command = code = ""
    parts.clear()


In [ ]:
# @title 完成 Drive 云端同步
import json, sys, subprocess
from pathlib import Path
from google.colab import drive
worker = Path("/content/fumiao-worker")
previous = -1
for attempt in range(3):
    plan = json.loads((worker / "state/storage-plan.json").read_text())
    persistent = [f for f in plan["files"] if f["tier"] == "persistent"]
    if persistent:
        print("等待模型上传到 Drive，完成后重新挂载……")
        drive.flush_and_unmount(timeout_ms=3_600_000)
        drive.mount("/content/drive")
        print(f"已同步 {len(persistent)} 个模型文件。")
    remaining = [f for f in plan["files"] if f["tier"] != "persistent"]
    if not remaining or len(persistent) == previous or attempt == 2:
        break
    previous = len(persistent)
    print("云端同步已完成，重试尚未持久保存的文件……")
    result = subprocess.run([sys.executable, str(worker / "cache_manager.py"), "--root", str(worker),
                             "--models", models, "--sync-only"], env=env)
    if result.returncode:
        raise RuntimeError("缓存补齐未完成；本地文件仍然保留，请检查上方日志")
plan = json.loads((worker / "state/storage-plan.json").read_text())
local_only = [f for f in plan["files"] if f["tier"] != "persistent"]
if local_only:
    print(f"接入已就绪，另有 {len(local_only)} 个文件仅在本地；下次启动需补齐。")
else:
    print("接入及 Drive 同步完成，可回网站生成。")
